# 01 — Exploratory Data Analysis & Dataset Preparation
**Project:** RoadWatch — Pothole Detection from Road Images using Deep Learning

This notebook covers:
1. Scanning raw and processed dataset splits (`train` / `val` / `test`).
2. Verifying zero data leakage across splits (using perceptual/MD5 image hashes).
3. Inspecting image resolutions, class balance, and bounding-box area distributions (to calibrate Low/Medium/High severity thresholds).

In [ ]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_ROOT = Path('../data/processed')
SPLITS = ['train', 'val', 'test']
print('Dataset root:', DATA_ROOT.resolve())

In [ ]:
def compute_md5(file_path: Path) -> str:
    h = hashlib.md5()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

records = []
box_ratios = []

for split in SPLITS:
    img_dir = DATA_ROOT / 'images' / split
    lbl_dir = DATA_ROOT / 'labels' / split
    if not img_dir.exists():
        continue
    for img_path in sorted(img_dir.glob('*.*')):
        if img_path.name == '.gitkeep':
            continue
        with Image.open(img_path) as im:
            w, h = im.size
        lbl_path = lbl_dir / f'{img_path.stem}.txt'
        n_boxes = 0
        if lbl_path.exists():
            lines = [ln.strip() for ln in lbl_path.read_text().splitlines() if ln.strip()]
            n_boxes = len(lines)
            for ln in lines:
                parts = [float(x) for x in ln.split()]
                if len(parts) == 5:
                    _, xc, yc, bw, bh = parts
                    box_ratios.append(bw * bh)
        records.append({
            'split': split,
            'filename': img_path.name,
            'width': w,
            'height': h,
            'num_potholes': n_boxes,
            'md5': compute_md5(img_path),
        })

df_meta = pd.DataFrame(records)
print(f'Total images indexed: {len(df_meta)}')
df_meta.head()

In [ ]:
# Verify Zero Data Leakage across train/val/test splits
if not df_meta.empty:
    dup_hashes = df_meta.groupby('md5')['split'].nunique()
    leaked = dup_hashes[dup_hashes > 1]
    print(f'Cross-split duplicate images (leakage check): {len(leaked)}')
    print(df_meta.groupby('split').agg(images=('filename', 'count'), potholes=('num_potholes', 'sum')))